### Cách dùng Notebook 01 hiện tại

1. Mở `01_train_val_test_batch_runner.ipynb`.
2. Chạy Cell 2 để:
- Import pipeline
- Khai báo toàn bộ cấu hình bạn muốn kiểm soát
- Khởi tạo đối tượng pipeline
3. Chạy Cell 4 để chạy toàn bộ train-val-test batch.
4. Sau khi chạy xong, Cell 4 sẽ:
- In đường dẫn thư mục batch run
- Đọc và hiển thị 2 bảng tổng hợp metrics/profiling.

Luồng chạy thực tế nằm ở `pipeline.py`.

Các config ảnh hưởng pipeline train như thế nào

1. CONFIG trong Cell 2
- batch_size: tăng thì nhanh hơn nhưng tốn VRAM/RAM hơn.
- epochs: số epoch tối đa cho nhánh pytorch.
- lr: learning rate cho optimizer.
- weight_decay: regularization cho pytorch.
- lr_factor + lr_patience: giảm learning rate khi MCC val không cải thiện.
- early_stop_patience: dừng sớm nếu MCC val không tăng.
- num_workers: số worker DataLoader.

2. MODELS_SPACE trong Cell 2
- Xác định tập model DNA và Protein để vét cạn tổ hợp ở Stage 1.
- Pipeline tự tách model theo seq_type dna/protein.
- Nếu thêm/bớt model ở đây, số tổ hợp train tăng/giảm trực tiếp.

3. EXPERIMENTS trong Cell 2
- Quy định bạn chạy kiểu nào:
- pytorch: train end-to-end mạng fusion.
- hybrid: train nhanh mạng pytorch để lấy f_global, sau đó train XGBoost.
- xgboost_pure: PCA + XGBoost trên đặc trưng không gian bảng/biến thể.
- Với ablation chỉ có 1 nhánh chuỗi, pipeline tự bỏ các fusion cần 2 nhánh như cross_attention/transformer/gating.

4. datasets trong Cell 2
- Mỗi phần tử định nghĩa 1 job train/val/test.
- test split ở đây còn ảnh hưởng E2E profiling vì Module 5 sẽ đọc FM profiling đúng theo split test tương ứng.

5. pooling_strategies trong Cell 2
- Chạy lặp theo từng pooling center/cls/mean.
- Mỗi pooling sẽ có leaderboard và best pair search riêng.

6. EXPLAINABILITY trong Cell 2
- enable_shap: bật SHAP cho các nhánh dùng XGBoost.
- enable_lime: bật LIME cho local explanation.
- max_background_samples: số mẫu nền tối đa cho SHAP/LIME.
- max_explain_samples: số mẫu test giải thích bằng SHAP.
- max_lime_samples: số mẫu test giải thích bằng LIME.
- lime_num_features: số feature rule mỗi mẫu cho LIME.
- random_state: seed cho sampling explainability.

### Output sau khi chạy xong

Sau khi chạy toàn bộ Notebook 01 (Cell 2 rồi Cell 4), output sẽ gồm các nhóm sau trong thư mục batch run mới:

1. Output tổng hợp cấp batch run
- experiments / `batch_run_YYYYMMDD_HHMM/`
- `global_leaderboard_metrics.csv`: metrics của model mình (toàn bộ run trong pipeline)
- `global_leaderboard_profiling.csv`: profiling E2E của model mình
- `global_compare_ours_vs_sota.csv`: bảng gộp so sánh `OURS` và `SOTA`
- `sota_metrics.csv`: metrics SOTA tính trực tiếp từ cột score/rankscore/pred
- `sota_column_mapping.csv`: mapping model SOTA -> cột nào được dùng
- `sota_required_columns_audit.csv`: audit đủ cột/thiếu cột/null theo từng dataset test

2. Output theo từng experiment con
Mỗi tổ hợp dataset + pooling + ablation + model + network sẽ có 1 thư mục con, ví dụ:
- `Train1Val_Test_center_dna_prot_nt_v1_500m_esm1b_650m_PyTorch_Concat/`

Bên trong thường có:
- `test_probabilities.csv`: dự đoán chi tiết theo Variant_ID
- `tensorboard_logs/`: log train/val
- `checkpoints/best_model.pth`: với nhánh PyTorch

3. Output explainability (nếu bật trong EXPLAINABILITY)
Trong các run dùng XGBoost (`hybrid`, `xgboost_pure`) sẽ có:
- `explainability/shap_global_importance.csv`
- `explainability/shap_local_top_ranked.csv`
- `explainability/lime_local_explanations.csv` (chỉ có khi `enable_lime=True`)

4. Output hiển thị ngay trên notebook (không phải file)
- In đường dẫn batch run
- In số dòng của metrics/profiling
- `display(df_compare.head(...))` cho bảng so sánh OURS vs SOTA
- `display(mapping/audit head(...))` để xem nhanh mapping cột và lỗi audit (nếu có)

Các đầu vào mà Notebook 01 kỳ vọng

- Bio normalized: dưới D:/variant_data/processed_parquet
- Geometry: dưới D:/variant_data/geometry/split
- Embedding pt: dưới D:/variant_data/fm_embeddings/split
- FM profiling json: `fm_profiling.json`

### SET UP

In [1]:
import os
import sys
import importlib
import torch
import pandas as pd

sys.path.append(os.path.abspath("../"))

try:
    import thop
    print(f"[*] thop version: {getattr(thop, '__version__', 'unknown')}")
except Exception as e:
    print(f"[CẢNH BÁO] thop import lỗi: {e}")

import core.module05_fusion_classifier.dataset as dataset_module
importlib.reload(dataset_module)
import core.module05_fusion_classifier.evaluator_profiler as evalprof_module
importlib.reload(evalprof_module)
import core.module05_fusion_classifier.pipeline as pipeline_module
importlib.reload(pipeline_module)
from core.module05_fusion_classifier.pipeline import FusionBatchPipeline
from core.module05_fusion_classifier.sota_benchmark import (
    evaluate_sota_from_test_files,
    SOTA_MODEL_COLUMNS_FULL,
    SOTA_REQUIRED_COLUMNS,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Device: {DEVICE}")

BASE_DIR = "D:/variant_data"
FM_PROFILE_JSON = f"{BASE_DIR}/profiling/fm_profiling.json"
GEOM_PROFILE_JSON = f"{BASE_DIR}/profiling/geom_profiling.json"

# Default config dua ve notebook de de kiem soat
CONFIG = {
    "batch_size": 256,
    "epochs": 30,
    "lr": 1e-4,
    "weight_decay": 1e-3,
    "lr_factor": 0.5,
    "lr_patience": 3,
    "early_stop_patience": 6,
    "num_workers": 0,
}

MODELS_SPACE = [
    {"name": "nt_v1_500m", "seq_type": "dna"},
    {"name": "nt_v3_650m", "seq_type": "dna"},
    {"name": "nt_v2_500m", "seq_type": "dna"},
    {"name": "esm1b_650m", "seq_type": "protein"},
    {"name": "esm2_650m", "seq_type": "protein"},
    {"name": "esmc_600m", "seq_type": "protein"},
]

EXPERIMENTS = [
    {"name": "PyTorch_Concat", "type": "pytorch", "fusion": "concat"},
    {"name": "PyTorch_CrossAttn", "type": "pytorch", "fusion": "cross_attention"},
    {"name": "PyTorch_Transformer", "type": "pytorch", "fusion": "transformer"},
    {"name": "PyTorch_Gating", "type": "pytorch", "fusion": "gating"},
    {"name": "Pure_XGBoost_Concat", "type": "xgboost_pure", "fusion": "concat"},
    {"name": "Hybrid_Concat_XGBoost", "type": "hybrid", "fusion": "concat"},
    {"name": "Hybrid_CrossAttn_XGBoost", "type": "hybrid", "fusion": "cross_attention"},
    {"name": "Hybrid_Transformer_XGBoost", "type": "hybrid", "fusion": "transformer"},
    {"name": "Hybrid_Gating_XGBoost", "type": "hybrid", "fusion": "gating"},
]

# Dataset split run theo du lieu hien co: train1/val va 4 tap test
datasets = [
    {"name": "Train1Val_Test", "train": "train1", "val": "val", "test": "test"},
    {"name": "Train1Val_ClinVarHQ", "train": "train1", "val": "val", "test": "clinvarhq"},
    {"name": "Train1Val_UniProt", "train": "train1", "val": "val", "test": "uniprot"},
    {"name": "Train1Val_ProteinGym", "train": "train1", "val": "val", "test": "proteingym"},
]

pooling_strategies = ["center", "cls", "mean"]

# Explainability: nen bat SHAP truoc, LIME de sau vi ton thoi gian hon
EXPLAINABILITY = {
    "enable_shap": True,
    "enable_lime": False,
    "max_background_samples": 512,
    "max_explain_samples": 200,
    "lime_num_features": 20,
    "max_lime_samples": 20,
    "random_state": 42,
}

# [MOI] SOTA benchmark tu cot score/rankscore/pred trong 4 test files
SOTA_TEST_FILES = {
    "test": f"{BASE_DIR}/test_full_seq_after_vep_final.parquet",
    "clinvarhq": f"{BASE_DIR}/clinvarhq_full_seq_after_vep_final.parquet",
    "uniprot": f"{BASE_DIR}/uniprot_full_seq_after_vep_final.parquet",
    "proteingym": f"{BASE_DIR}/proteingym_full_seq_after_vep_final.parquet",
}
SOTA_LABEL_COL = None
SOTA_LABEL_CANDIDATES = ["Pathogenicity_Label", "Label"]
SOTA_THRESHOLD = 0.5

# Dung bo cot day du cho SOTA theo danh sach chuan
SOTA_MODEL_COLUMNS = SOTA_MODEL_COLUMNS_FULL
SOTA_REQUIRED = SOTA_REQUIRED_COLUMNS
SOTA_ENFORCE_REQUIRED = True

# Resume control: dat True de tiep tuc 1 batch run dang do
RESUME_ENABLED = True
RESUME_BATCH_RUN_DIR = '../experiments/batch_run_20260902_1626' # vd: '../experiments/batch_run_20260902_1430'

pipeline = FusionBatchPipeline(
    base_dir=BASE_DIR,
    config=CONFIG,
    datasets=datasets,
    models_space=MODELS_SPACE,
    pooling_strategies=pooling_strategies,
    experiments=EXPERIMENTS,
    explainability=EXPLAINABILITY,
    fm_profile_json=FM_PROFILE_JSON,
    geom_profile_json=GEOM_PROFILE_JSON,
    batch_run_dir=RESUME_BATCH_RUN_DIR,
)

print(f"[*] FM profiling json: {FM_PROFILE_JSON}")
print(f"[*] Geometry profiling json: {GEOM_PROFILE_JSON}")
print(f"[*] Batch run dir: {pipeline.batch_run_dir}")

[*] thop version: 0.1.1
[*] Device: cpu
[*] FM profiling json: D:/variant_data/profiling/fm_profiling.json
[*] Geometry profiling json: D:/variant_data/profiling/geom_profiling.json
[*] Batch run dir: d:\Missense_variant_pathogenicity_prediction\experiments\batch_run_20260902_1626


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### The Master Loop

In [2]:
# 1) Benchmark SOTA truc tiep tu cot score/rankscore/pred tren 4 test files
sota_result = evaluate_sota_from_test_files(
    test_files=SOTA_TEST_FILES,
    output_dir=pipeline.batch_run_dir,
    label_col=SOTA_LABEL_COL,
    label_candidates=SOTA_LABEL_CANDIDATES,
    model_columns=SOTA_MODEL_COLUMNS,
    threshold=SOTA_THRESHOLD,
    required_columns=SOTA_REQUIRED,
    enforce_required_columns=SOTA_ENFORCE_REQUIRED,
)
print(f"[SOTA] Metrics CSV: {sota_result['metrics_path']} | rows={sota_result['num_rows']}")
print(f"[SOTA] Mapping CSV: {sota_result['mapping_path']} | unique models={sota_result['num_models']}")
if sota_result.get('audit') is not None:
    audit = sota_result['audit']
    print(f"[SOTA] Audit CSV: {audit['audit_path']}")
    print(f"[SOTA] missing_count={audit['missing_count']} | null_issue_count={audit['null_issue_count']}")

# 2) Chay pipeline model cua chung ta
result = pipeline.run(resume=RESUME_ENABLED)

print("\n" + "=" * 80)
print("[THANH CONG] Module 5 batch runner da hoan tat")
print("=" * 80)
print(f"Batch run dir: {result['batch_run_dir']}")
print(f"Metrics CSV: {result['metrics_path']} | rows={result['num_rows_metrics']}")
print(f"Profiling CSV: {result['profiling_path']} | rows={result['num_rows_profiling']}")

# 3) Gop bang so sanh SOTA vs Ours
ours = pd.read_csv(result["metrics_path"])
sota = pd.read_csv(sota_result["metrics_path"])
ours["Source"] = "OURS"
sota["Source"] = "SOTA"

df_compare = pd.concat([ours, sota], ignore_index=True)
df_compare = df_compare.sort_values(by=["Dataset", "MCC"], ascending=[True, False])
df_compare.to_csv(f"{result['batch_run_dir']}/global_compare_ours_vs_sota.csv", index=False)

print(f"Compare CSV: {result['batch_run_dir']}/global_compare_ours_vs_sota.csv")
display(df_compare.head(40))
display(pd.read_csv(sota_result["mapping_path"]).head(40))
if sota_result.get('audit') is not None:
    display(pd.read_csv(sota_result['audit']['audit_path']).query('exists == False or null_count > 0').head(100))

[SOTA] Metrics CSV: d:\Missense_variant_pathogenicity_prediction\experiments\batch_run_20260902_1626/sota_metrics.csv | rows=112
[SOTA] Mapping CSV: d:\Missense_variant_pathogenicity_prediction\experiments\batch_run_20260902_1626/sota_column_mapping.csv | unique models=28
[SOTA] Audit CSV: d:\Missense_variant_pathogenicity_prediction\experiments\batch_run_20260902_1626/sota_required_columns_audit.csv
[SOTA] missing_count=0 | null_issue_count=0
[*] Device: cpu
[*] Batch run dir: d:\Missense_variant_pathogenicity_prediction\experiments\batch_run_20260902_1626
[*] Resume mode: loaded metrics=1548 rows, profiling=1548 rows

[*] POOLING: CENTER
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio']
[+] Dataset sẵn sàng: 104385 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio']
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio']
[+] Dataset sẵn sàng: 6095 mẫu. Đã tối ưu hóa 

[Epoch 01] train_loss=0.2879 | val_loss=0.2291 | val_mcc=0.6917 | val_f1=0.7430 | val_auroc=0.9471


[Epoch 02] train_loss=0.2598 | val_loss=0.2301 | val_mcc=0.6802 | val_f1=0.7273 | val_auroc=0.9486


[Epoch 03] train_loss=0.2561 | val_loss=0.2251 | val_mcc=0.6838 | val_f1=0.7357 | val_auroc=0.9487


[Epoch 04] train_loss=0.2530 | val_loss=0.2240 | val_mcc=0.6937 | val_f1=0.7472 | val_auroc=0.9488


...

[Epoch 12] train_loss=0.0574 | val_loss=0.9809 | val_mcc=0.3413 | val_f1=0.2530 | val_auroc=0.9215


[Epoch 13] train_loss=0.0531 | val_loss=1.0876 | val_mcc=0.3150 | val_f1=0.2216 | val_auroc=0.9202
[EarlyStop] epoch=13 | best_val_mcc=0.4605
[Test] train1__val_mean_bio_geom_prot_nt_v3_650m_esm1b_650m_PyTorch_Concat | test=Train1Val_Test | MCC=0.4630 | F1=0.4393 | AUROC=0.9505 | AUPRC=0.9007
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['prot', 'bio', 'geom']
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 104385 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['prot', 'bio', 'geom']
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['prot', 'bio', 'geom']
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 703 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Running config=train1__val_mean_bio_geom_prot_nt_v3_650m_esm1b_650m_PyTorch_Concat | test=Train1Val_ClinVarHQ
[*] Reuse trained PyTorch 

,Dataset,Pooling,Ablation,DNA_Model,Prot_Model,Network,Accuracy,Precision,Recall,Specificity,F1_Score,MCC,AUROC,AUPRC,Source
0,Train1Val_ClinVarHQ,cls,bio_dna_prot,nt_v3_650m,esm1b_650m,Pure_XGBoost_Concat,0.9275,0.9744,0.8764,0.9775,0.9228,0.8590,0.9863,0.9874,OURS
1,Train1Val_ClinVarHQ,center,bio_dna_prot,nt_v3_650m,esmc_600m,Pure_XGBoost_Concat,0.9260,0.9744,0.8736,0.9775,0.9212,0.8564,0.9881,0.9885,OURS
2,Train1Val_ClinVarHQ,mean,bio_dna_prot,nt_v3_650m,esm1b_650m,Hybrid_Concat_XGBoost,0.9260,0.9684,0.8793,0.9718,0.9217,0.8555,0.9827,0.9835,OURS
3,Train1Val_ClinVarHQ,mean,bio_dna,nt_v3_650m,NaN,Pure_XGBoost_Concat,0.9246,0.9836,0.8621,0.9859,0.9188,0.8555,0.9840,0.9856,OURS
4,Train1Val_ClinVarHQ,center,bio_dna_prot,nt_v3_650m,esmc_600m,Hybrid_CrossAttn_XGBoost,0.9260,0.9596,0.8879,0.9634,0.9224,0.8543,0.9863,0.9865,OURS
5,Train1Val_ClinVarHQ,center,bio_dna_prot,nt_v3_650m,esmc_600m,Hybrid_Gating_XGBoost,0.9246,0.9712,0.8736,0.9746,0.9198,0.8533,0.9836,0.9847,OURS
6,Train1Val_ClinVarHQ,center,bio_dna_prot,nt_v3_650m,esmc_600m,Hybrid_Concat_XGBoost,0.9246,0.9595,0.8851,0.9634,0.9208,0.8516,0.9865,0.9864,OURS
7,Train1Val_ClinVarHQ,center,bio_dna_prot,nt_v3_650m,esmc_600m,Hybrid_Transformer_XGBoost,0.9232,0.9742,0.8678,0.9775,0.9179,0.8512,0.9840,0.9847,OURS
8,Train1Val_ClinVarHQ,mean,bio_dna_prot,nt_v3_650m,esm1b_650m,Hybrid_Gating_XGBoost,0.9232,0.9712,0.8707,0.9746,0.9182,0.8507,0.9822,0.9825,OURS
9,Train1Val_ClinVarHQ,center,bio_prot,NaN,esmc_600m,Hybrid_Concat_XGBoost,0.9232,0.9652,0.8764,0.9690,0.9187,0.8498,0.9824,0.9842,OURS


,Dataset,Model,file_path,score_col,rankscore_col,pred_col,resolved_prob_col,resolved_prob_kind,resolved_threshold,pred_source,label_col
0,test,SIFT,D:/variant_data/test_full_seq_after_vep_final....,['SIFT_score'],"['SIFT_converted_rankscore', 'SIFT_rankscore']",['SIFT_pred'],SIFT_converted_rankscore,rankscore,0.395750,SIFT_pred,Label
1,test,SIFT4G,D:/variant_data/test_full_seq_after_vep_final....,['SIFT4G_score'],"['SIFT4G_converted_rankscore', 'SIFT4G_ranksco...",['SIFT4G_pred'],SIFT4G_converted_rankscore,rankscore,0.395750,SIFT4G_pred,Label
2,test,Polyphen2_HDIV,D:/variant_data/test_full_seq_after_vep_final....,['Polyphen2_HDIV_score'],['Polyphen2_HDIV_rankscore'],['Polyphen2_HDIV_pred'],Polyphen2_HDIV_rankscore,rankscore,0.380280,Polyphen2_HDIV_pred,Label
3,test,Polyphen2_HVAR,D:/variant_data/test_full_seq_after_vep_final....,['Polyphen2_HVAR_score'],['Polyphen2_HVAR_rankscore'],['Polyphen2_HVAR_pred'],Polyphen2_HVAR_rankscore,rankscore,0.487620,Polyphen2_HVAR_pred,Label
4,test,MutationTaster,D:/variant_data/test_full_seq_after_vep_final....,['MutationTaster_score'],['MutationTaster_rankscore'],['MutationTaster_pred'],MutationTaster_rankscore,rankscore,0.317330,MutationTaster_pred,Label
5,test,MetaSVM,D:/variant_data/test_full_seq_after_vep_final....,['MetaSVM_score'],['MetaSVM_rankscore'],['MetaSVM_pred'],MetaSVM_rankscore,rankscore,0.822570,MetaSVM_pred,Label
6,test,MetaLR,D:/variant_data/test_full_seq_after_vep_final....,['MetaLR_score'],['MetaLR_rankscore'],['MetaLR_pred'],MetaLR_rankscore,rankscore,0.811010,MetaLR_pred,Label
7,test,MetaRNN,D:/variant_data/test_full_seq_after_vep_final....,['MetaRNN_score'],['MetaRNN_rankscore'],['MetaRNN_pred'],MetaRNN_rankscore,rankscore,0.614900,MetaRNN_pred,Label
8,test,M-CAP,D:/variant_data/test_full_seq_after_vep_final....,['M-CAP_score'],['M-CAP_rankscore'],['M-CAP_pred'],M-CAP_rankscore,rankscore,0.025000,M-CAP_pred,Label
9,test,REVEL,D:/variant_data/test_full_seq_after_vep_final....,['REVEL_score'],['REVEL_rankscore'],['REVEL_pred'],REVEL_rankscore,rankscore,0.500000,REVEL_pred,Label


,Dataset,file_path,column,exists,null_count
